# EQL product coefficients

This notebook shows how `EQL.product_coefficients(...)` expands product neurons back into monomials over the original feature basis.

The example below uses a tiny hand-set model so the coefficients are easy to inspect.


In [1]:
import os
os.getcwd()
os.chdir("../")
os.getcwd()

'c:\\Users\\sami\\OneDrive\\Documents\\PDENets'

In [2]:
import torch
from prog.mlps import EQL

def poly_to_string(poly):
    pieces = []
    for monomial, coeff in sorted(poly.items(), key=lambda item: (len(item[0]), item[0])):
        term = "*".join(monomial) if monomial else "1"
        pieces.append(f"{coeff:+.3f}?{term}")
    return " ".join(pieces) if pieces else "0"

In [8]:
# Build a tiny EQL model with three primitive inputs.
# The product layer has two linear factors, so p0 becomes a quadratic polynomial.
feature_names = ["u", "u_x", "u_xx", "u_xxx"]
model = EQL(in_dim=len(feature_names), prod_dim=2, num_layers=2, bias=False)


In [9]:
model.update_signal

'coo'

In [10]:
model.linears,model.readout


(ModuleList(
   (0): Linear(in_features=4, out_features=2, bias=False)
   (1): Linear(in_features=4, out_features=1, bias=False)
 ),
 Linear(in_features=5, out_features=1, bias=False))

In [12]:

with torch.no_grad():
    model.linears[0].weight.copy_(torch.tensor([[1.0, 2.0, 0.0, 0.0],
                                               [0.0, 1.0, 1.0, 0.0]])), 
    model.linears[1].weight.copy_(torch.tensor([[1.0, 2.0, 0.0, 0.0]]))
    model.readout.weight.copy_(torch.tensor([[0.5, -1.0, 0.25, 1.0, 3.0]]))

raw_products = model.product_coefficients(feature_names=feature_names, include_readout=False)
weighted_products = model.product_coefficients(feature_names=feature_names, include_readout=True)

raw_products, weighted_products
 

({'p0': {('u', 'u_x'): 1.0,
   ('u', 'u_xx'): 1.0,
   ('u_x', 'u_x'): 2.0,
   ('u_x', 'u_xx'): 2.0},
  'p1': {('u', 'u_x', 'u'): 1.0,
   ('u', 'u_x', 'u_x'): 2.0,
   ('u', 'u_xx', 'u'): 1.0,
   ('u', 'u_xx', 'u_x'): 2.0,
   ('u_x', 'u_x', 'u'): 2.0,
   ('u_x', 'u_x', 'u_x'): 4.0,
   ('u_x', 'u_xx', 'u'): 2.0,
   ('u_x', 'u_xx', 'u_x'): 4.0}},
 {'p0': {('u', 'u_x'): 1.0,
   ('u', 'u_xx'): 1.0,
   ('u_x', 'u_x'): 2.0,
   ('u_x', 'u_xx'): 2.0},
  'p1': {('u', 'u_x', 'u'): 3.0,
   ('u', 'u_x', 'u_x'): 6.0,
   ('u', 'u_xx', 'u'): 3.0,
   ('u', 'u_xx', 'u_x'): 6.0,
   ('u_x', 'u_x', 'u'): 6.0,
   ('u_x', 'u_x', 'u_x'): 12.0,
   ('u_x', 'u_xx', 'u'): 6.0,
   ('u_x', 'u_xx', 'u_x'): 12.0}})

In [7]:
print("Raw product polynomial:")
print(poly_to_string(raw_products["p0"]))

print("\nContribution after readout weighting:")
print(poly_to_string(weighted_products["p0"]))

print("\nInterpretation:")
print("p0 = (u + 2 u_x) (u_x + u_xx)")
print("include_readout=True multiplies p0 by the final readout weight for that product neuron.")


Raw product polynomial:
+1.000?u*u_x +1.000?u*u_xx +2.000?u_x*u_x +2.000?u_x*u_xx

Contribution after readout weighting:
+1.000?u*u_x +1.000?u*u_xx +2.000?u_x*u_x +2.000?u_x*u_xx

Interpretation:
p0 = (u + 2 u_x) (u_x + u_xx)
include_readout=True multiplies p0 by the final readout weight for that product neuron.
